# Crowd Head-ID Tracking — Self-Contained Colab T4 Handoff (v7)

This notebook is **fully self-contained** — there is **NO git clone**.
The pipeline source (`src/vision/__init__.py`, `src/vision/detector.py`,
`scripts/run_head_id_stability.py`) is written into this Colab session
by the `%%writefile` cells below, then executed directly.

## Quick start
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (≈ 5–10 min for the full video, ~1 min for --max-frames 200)

## Reference numbers
| arm | confirmed_unique | inflation_factor |
|-----|-----------------|------------------|
| baseline | ~338 | ~3.1× |
| **fix_0p16_wide_guard_app (v7 winner)** | **~125** | **~3.1×** |
| A100 best (same model/config) | **105** | **2.5×** |

Generated by `scripts/build_handoff_notebook.py` from commit `c992dc0` (+uncommitted changes) at 2026-06-15T05:58:37Z.


In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then Run all."
print("GPU:", torch.cuda.get_device_name(0))


## Step 1 — write the pipeline source into this Colab session (no repo clone)


In [ ]:
import os
os.makedirs("src/vision", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
print("dirs ready")


In [ ]:
%%writefile src/vision/__init__.py
"""Core vision modules for detection, tracking, and crowd analytics."""


In [ ]:
%%writefile src/vision/detector.py
"""Person detector abstraction over Ultralytics YOLO models."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import time
from typing import Literal

try:
    from ultralytics import YOLO
except ModuleNotFoundError:  # pragma: no cover - environment-specific dependency
    YOLO = None  # type: ignore[assignment]

DEFAULT_MODEL_CANDIDATES: tuple[str, ...] = ("yolo11s.pt", "yolo11n.pt", "yolov8n.pt")
DEFAULT_PERSON_CLASS_ID = 0
FINE_TUNED_MODEL_PATH = Path("models/fine_tuned/best.pt")
DEFAULT_HEAD_MODEL_PATH = Path("models/fine_tuned/head_detector/weights/best.pt")
DetectorMode = Literal["body", "head", "hybrid"]

# Readable per-detection labels used by hybrid fusion so reports/overlays can
# distinguish the detection source even though they are tracked as one stream.
BODY_PASSENGER_LABEL = "body/passenger"
HEAD_PASSENGER_LABEL = "head/passenger"


def normalize_detector_mode(detector_mode: str) -> DetectorMode:
    """Validate and normalize detector mode strings."""
    normalized = detector_mode.strip().lower()
    if normalized not in {"body", "head", "hybrid"}:
        raise ValueError("detector_mode must be one of 'body', 'head', or 'hybrid'")
    return normalized  # type: ignore[return-value]


def default_model_for_mode(detector_mode: str, configured_body_model: str | None = None) -> str:
    """Return the default model path/name for a detector mode."""
    mode = normalize_detector_mode(detector_mode)
    if mode == "head":
        return str(DEFAULT_HEAD_MODEL_PATH)
    return configured_body_model or DEFAULT_MODEL_CANDIDATES[1]


def class_name_for_mode(detector_mode: str, model_class_name: str | None = None) -> str:
    """Return the label exposed to analytics and overlays for a detector mode."""
    mode = normalize_detector_mode(detector_mode)
    if mode == "head":
        return HEAD_PASSENGER_LABEL
    return "person/passenger" if model_class_name in (None, "", "person") else model_class_name


def hybrid_class_name_for_source(source_type: str) -> str:
    """Return the readable fused label for a hybrid detection source."""
    return HEAD_PASSENGER_LABEL if source_type == "head" else BODY_PASSENGER_LABEL


@dataclass(slots=True)
class NormalizedDetection:
    """Normalized frame-level detection or track output."""

    track_id: int | None
    bbox: tuple[float, float, float, float]
    confidence: float
    class_name: str
    frame_index: int
    timestamp: float
    detector_mode: DetectorMode = "body"
    # Source of the detection within a fused/hybrid stream ("body" or "head").
    # Defaults to None for single-mode pipelines, where detector_mode already
    # encodes the source. Reports/annotations use this to label fused output.
    source_type: str | None = None


@dataclass(slots=True)
class DetectionResult:
    """Normalized detection collection for one frame."""

    detections: list[NormalizedDetection]


def resolve_model_candidates(
    weights_path: str | None = None,
    *,
    accuracy_weights: str | None = None,
    legacy_fallback_weights: str | None = None,
    use_fine_tuned_if_available: bool = True,
) -> tuple[str, ...]:
    """Return ordered model candidates from explicit and fallback settings."""
    candidates: list[str] = []
    if weights_path:
        candidates.append(weights_path)
    if use_fine_tuned_if_available and FINE_TUNED_MODEL_PATH.exists():
        candidates.append(str(FINE_TUNED_MODEL_PATH))
    if accuracy_weights:
        candidates.append(accuracy_weights)
    candidates.extend(DEFAULT_MODEL_CANDIDATES)
    if legacy_fallback_weights:
        candidates.append(legacy_fallback_weights)

    ordered: list[str] = []
    seen: set[str] = set()
    for candidate in candidates:
        normalized = str(candidate).strip()
        if not normalized or normalized in seen:
            continue
        seen.add(normalized)
        ordered.append(normalized)
    return tuple(ordered)


def build_model(
    weights_path: str | None = None,
    *,
    accuracy_weights: str | None = None,
    legacy_fallback_weights: str | None = None,
    use_fine_tuned_if_available: bool = True,
) -> YOLO:
    """Create a YOLO model, falling back to known lightweight defaults."""
    if YOLO is None:
        raise RuntimeError(
            "Ultralytics is not installed. Install dependencies first: pip install -r requirements.txt"
        )
    candidates = resolve_model_candidates(
        weights_path,
        accuracy_weights=accuracy_weights,
        legacy_fallback_weights=legacy_fallback_weights,
        use_fine_tuned_if_available=use_fine_tuned_if_available,
    )
    last_error: Exception | None = None
    for candidate in candidates:
        try:
            return YOLO(candidate)
        except Exception as exc:  # pragma: no cover - depends on runtime/model availability
            last_error = exc
    tried = ", ".join(candidates)
    raise RuntimeError(
        f"Unable to load an Ultralytics YOLO model. Tried: {tried}. "
        "Pass an existing local weights file via --model (e.g. --model yolo11n.pt), "
        "or allow network access so Ultralytics can download the weights on first run. "
        f"Underlying error: {last_error}"
    ) from last_error


class Detector:
    """YOLO detector that returns normalized person detections."""

    def __init__(
        self,
        weights_path: str | None = None,
        device: str = "cpu",
        confidence: float = 0.35,
        iou: float = 0.5,
        person_class_id: int = DEFAULT_PERSON_CLASS_ID,
        imgsz: int = 640,
        augment: bool = False,
        max_det: int = 300,
        half: bool = False,
        accuracy_weights: str | None = None,
        legacy_fallback_weights: str | None = None,
        use_fine_tuned_if_available: bool = True,
        detector_mode: str = "body",
        class_name_override: str | None = None,
    ) -> None:
        """Initialize the detector with model and inference parameters."""
        self.detector_mode = normalize_detector_mode(detector_mode)
        self.device = device
        self.confidence = confidence
        self.iou = iou
        self.person_class_id = person_class_id
        self.imgsz = imgsz
        self.augment = augment
        self.max_det = max_det
        self.half = half
        self.weights_path = weights_path
        self.class_name_override = class_name_override
        self.model = build_model(
            weights_path,
            accuracy_weights=accuracy_weights,
            legacy_fallback_weights=legacy_fallback_weights,
            use_fine_tuned_if_available=use_fine_tuned_if_available,
        )

    def detect(
        self,
        frame: object,
        frame_index: int = 0,
        timestamp: float | None = None,
    ) -> DetectionResult:
        """Run person detection on one frame and normalize outputs."""
        frame_ts = time() if timestamp is None else timestamp
        predictions = self.model.predict(
            source=frame,
            conf=self.confidence,
            iou=self.iou,
            classes=[self.person_class_id],
            device=self.device,
            imgsz=self.imgsz,
            augment=self.augment,
            max_det=self.max_det,
            half=self.half,
            verbose=False,
        )
        if not predictions:
            return DetectionResult(detections=[])

        prediction = predictions[0]
        boxes = prediction.boxes
        if boxes is None:
            return DetectionResult(detections=[])

        names = prediction.names or getattr(self.model, "names", {})
        normalized: list[NormalizedDetection] = []
        for box in boxes:
            class_id = int(box.cls.item())
            if class_id != self.person_class_id:
                continue
            x1, y1, x2, y2 = (float(value) for value in box.xyxy[0].tolist())
            class_name = self.class_name_override or class_name_for_mode(
                self.detector_mode,
                str(names.get(class_id, "person")),
            )
            normalized.append(
                NormalizedDetection(
                    track_id=None,
                    bbox=(x1, y1, x2, y2),
                    confidence=float(box.conf.item()),
                    class_name=class_name,
                    frame_index=frame_index,
                    timestamp=frame_ts,
                    detector_mode=self.detector_mode,
                )
            )
        return DetectionResult(detections=normalized)


In [ ]:
%%writefile scripts/run_head_id_stability.py
"""Run head-only ID-stability experiments on a video.

This is the script version of ``railway_head_id_stability_colab.ipynb``. It is
intended for Colab/A100 and local smoke tests: detect heads, track them with a
tuned ByteTrack wrapper, optionally stitch short track fragments, and write
honest ID-stability metrics plus annotated videos.
"""

from __future__ import annotations

import argparse
import inspect
import json
import math
import shutil
import sys
import urllib.request
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.vision.detector import Detector, NormalizedDetection

try:
    import supervision as sv
except Exception:  # pragma: no cover - optional runtime fallback
    sv = None  # type: ignore[assignment]


SAMPLE_VIDEO_URL = "https://videos.pexels.com/video-files/12049569/12049569-hd_1280_720_25fps.mp4"
DEFAULT_RPEE_WEIGHTS = Path("models/fine_tuned/head_detector_s/weights/best.pt")
DEFAULT_CROWDHUMAN_WEIGHTS = Path("crowdhuman_head_s_best.pt")


@dataclass(frozen=True)
class Arm:
    """One experiment arm."""

    name: str
    conf: float
    activation: float
    consec: int
    expand: bool
    stitch: bool
    stitch_gap_frames: int | None = None
    stitch_dist_heads: float | None = None
    stitch_mode: str | None = None
    stitch_ambiguity_ratio: float | None = None
    stitch_max_speed_heads: float | None = None
    stitch_appearance_weight: float | None = None
    stitch_max_appearance_cost: float | None = None
    stitch_direction_weight: float | None = None
    stitch_max_direction_cost: float | None = None
    stitch_max_jump_heads: float | None = None


@dataclass(frozen=True)
class StabilityConfig:
    """ID-stability knobs shared across arms."""

    imgsz: int = 1536
    detector_iou: float = 0.55
    head_nms_iou: float = 0.60
    max_det: int = 1000
    min_confirmed_age: int = 3
    lost_track_buffer: int = 90
    match_thresh: float = 0.85
    box_expand_factor: float = 1.6
    stitch_gap_frames: int = 45
    stitch_dist_heads: float = 1.5
    stitch_mode: str = "spatial"
    stitch_ambiguity_ratio: float = 0.80
    stitch_max_speed_heads: float = 0.45
    stitch_appearance_weight: float = 0.0
    stitch_max_appearance_cost: float = 1.0
    stitch_direction_weight: float = 0.0
    stitch_max_direction_cost: float = 1.0
    stitch_max_jump_heads: float = 0.0
    switch_lookback: int = 45


DEFAULT_ARMS = (
    Arm("baseline", conf=0.16, activation=0.45, consec=1, expand=False, stitch=False),
    Arm("fix_0p16", conf=0.16, activation=0.20, consec=3, expand=True, stitch=True),
    Arm("fix_0p30", conf=0.30, activation=0.30, consec=3, expand=True, stitch=True),
    Arm(
        "fix_0p16_safe",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=120,
        stitch_dist_heads=2.5,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.65,
        stitch_max_speed_heads=0.35,
    ),
    Arm(
        "fix_0p16_loose",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=90,
        stitch_dist_heads=3.0,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.80,
        stitch_max_speed_heads=0.45,
    ),
    Arm(
        "fix_0p16_loose_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=120,
        stitch_dist_heads=3.4,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.78,
        stitch_max_speed_heads=0.45,
        stitch_appearance_weight=0.35,
        stitch_max_appearance_cost=0.72,
    ),
    Arm(
        "fix_0p16_loose_app_min",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=150,
        stitch_dist_heads=4.0,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.88,
        stitch_max_speed_heads=0.60,
        stitch_appearance_weight=0.25,
        stitch_max_appearance_cost=0.80,
    ),
    Arm(
        "fix_0p16_loose_oc",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=130,
        stitch_dist_heads=3.6,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.82,
        stitch_max_speed_heads=0.50,
        stitch_direction_weight=0.15,
        stitch_max_direction_cost=0.75,
    ),
    Arm(
        "fix_0p16_loose_oc_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=160,
        stitch_dist_heads=4.2,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.88,
        stitch_max_speed_heads=0.65,
        stitch_appearance_weight=0.25,
        stitch_max_appearance_cost=0.82,
        stitch_direction_weight=0.10,
        stitch_max_direction_cost=0.85,
    ),
    Arm(
        "fix_0p16_loose_guard",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=150,
        stitch_dist_heads=4.0,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.88,
        stitch_max_speed_heads=0.60,
        stitch_max_jump_heads=6.0,
    ),
    Arm(
        "fix_0p16_loose_guard_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=160,
        stitch_dist_heads=4.2,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.88,
        stitch_max_speed_heads=0.65,
        stitch_appearance_weight=0.20,
        stitch_max_appearance_cost=0.82,
        stitch_direction_weight=0.10,
        stitch_max_direction_cost=0.85,
        stitch_max_jump_heads=6.5,
    ),
    Arm(
        "fix_0p16_wide_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=150,
        stitch_dist_heads=4.0,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.88,
        stitch_max_speed_heads=0.65,
        stitch_appearance_weight=0.45,
        stitch_max_appearance_cost=0.65,
    ),
    Arm(
        "fix_0p16_wide_app_min",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=180,
        stitch_dist_heads=4.5,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.92,
        stitch_max_speed_heads=0.70,
        stitch_appearance_weight=0.35,
        stitch_max_appearance_cost=0.76,
    ),
    Arm(
        "fix_0p16_wide_oc",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=180,
        stitch_dist_heads=4.5,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.92,
        stitch_max_speed_heads=0.75,
        stitch_direction_weight=0.20,
        stitch_max_direction_cost=0.80,
    ),
    Arm(
        "fix_0p16_wide_oc_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=220,
        stitch_dist_heads=5.0,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.95,
        stitch_max_speed_heads=0.85,
        stitch_appearance_weight=0.30,
        stitch_max_appearance_cost=0.80,
        stitch_direction_weight=0.15,
        stitch_max_direction_cost=0.85,
    ),
    Arm(
        "fix_0p16_wide_guard",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=180,
        stitch_dist_heads=4.5,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=0.92,
        stitch_max_speed_heads=0.70,
        stitch_max_jump_heads=7.0,
    ),
    Arm(
        "fix_0p16_wide_guard_app",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=220,
        stitch_dist_heads=5.0,
        stitch_mode="observation",
        stitch_ambiguity_ratio=0.95,
        stitch_max_speed_heads=0.85,
        stitch_appearance_weight=0.20,
        stitch_max_appearance_cost=0.82,
        stitch_direction_weight=0.12,
        stitch_max_direction_cost=0.85,
        stitch_max_jump_heads=7.5,
    ),
    Arm(
        "fix_0p16_wide",
        conf=0.16,
        activation=0.20,
        consec=3,
        expand=True,
        stitch=True,
        stitch_gap_frames=150,
        stitch_dist_heads=4.0,
        stitch_mode="velocity",
        stitch_ambiguity_ratio=1.0,
        stitch_max_speed_heads=0.70,
    ),
)

MANUAL_GT = {80: 25, 240: 27, 400: 21, 560: 28, 720: 24}


def parse_args() -> argparse.Namespace:
    """Parse CLI args."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--video", type=Path, default=Path("data/input_videos/sample.mp4"))
    parser.add_argument(
        "--model",
        type=Path,
        default=DEFAULT_RPEE_WEIGHTS,
        help="Head detector weights. Default is the RPEE-trained head_detector_s model.",
    )
    parser.add_argument(
        "--model-label",
        default=None,
        help="Short label used in output folder names. Defaults to model stem/parent.",
    )
    parser.add_argument(
        "--arms",
        nargs="+",
        default=None,
        help="Arm names to run. With --sweep-profiles, generated sweep_* names are also valid.",
    )
    parser.add_argument(
        "--sweep-profiles",
        nargs="+",
        default=[],
        choices=["loose", "wide"],
        help="Append generated loose/wide stitching candidates for parameter search.",
    )
    parser.add_argument(
        "--sweep-limit",
        type=int,
        default=0,
        help="Limit generated sweep arms per profile for quick smoke tests. 0 = all.",
    )
    parser.add_argument("--output-root", type=Path, default=Path("data/outputs/head_id_stability"))
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--half", action="store_true")
    parser.add_argument("--imgsz", type=int, default=1536)
    parser.add_argument("--max-frames", type=int, default=0, help="0 = whole video")
    parser.add_argument(
        "--stitch-mode",
        choices=["spatial", "velocity", "observation"],
        default="spatial",
        help="Default stitching mode for arms without their own override.",
    )
    parser.add_argument("--download-sample", action="store_true")
    parser.add_argument(
        "--draw-annotated",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Write per-arm annotated videos. Disable for large metric sweeps.",
    )
    parser.add_argument("--make-side-by-side", action=argparse.BooleanOptionalAction, default=True)
    return parser.parse_args()


def sweep_arms(profile: str, *, limit: int = 0) -> list[Arm]:
    """Generate parameter-search arms around the strongest loose/wide candidates."""
    if profile == "loose":
        specs = [
            ("vel", 120, 3.6, 0.86, 0.55, 5.5, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("vel", 150, 4.0, 0.88, 0.60, 6.0, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("vel", 180, 4.4, 0.90, 0.68, 6.8, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("app", 150, 4.0, 0.88, 0.60, 6.0, 0.20, 0.82, 0.00, 1.00, "velocity"),
            ("obs", 160, 4.2, 0.88, 0.65, 6.5, 0.00, 1.00, 0.12, 0.85, "observation"),
            ("obsapp", 160, 4.2, 0.90, 0.65, 6.5, 0.20, 0.82, 0.10, 0.85, "observation"),
            ("obsapp", 190, 4.8, 0.92, 0.75, 7.2, 0.18, 0.84, 0.10, 0.88, "observation"),
        ]
    elif profile == "wide":
        specs = [
            ("vel", 150, 4.0, 1.00, 0.70, 0.0, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("velg", 180, 4.5, 0.92, 0.70, 7.0, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("velg", 220, 5.0, 0.95, 0.82, 7.5, 0.0, 1.00, 0.00, 1.00, "velocity"),
            ("appg", 180, 4.5, 0.92, 0.72, 7.0, 0.20, 0.82, 0.00, 1.00, "velocity"),
            ("obs", 180, 4.5, 0.92, 0.75, 7.2, 0.0, 1.00, 0.18, 0.85, "observation"),
            ("obsapp", 220, 5.0, 0.95, 0.85, 7.5, 0.18, 0.84, 0.12, 0.88, "observation"),
            ("obsapp", 260, 5.5, 0.97, 0.92, 8.0, 0.15, 0.86, 0.10, 0.90, "observation"),
        ]
    else:
        raise ValueError(f"Unknown sweep profile: {profile}")

    arms = [
        Arm(
            name=f"sweep_{profile}_{idx:02d}_{kind}_g{gap}_d{dist:g}_j{jump:g}",
            conf=0.16,
            activation=0.20,
            consec=3,
            expand=True,
            stitch=True,
            stitch_gap_frames=gap,
            stitch_dist_heads=dist,
            stitch_mode=mode,
            stitch_ambiguity_ratio=ambiguity,
            stitch_max_speed_heads=speed,
            stitch_appearance_weight=app_weight,
            stitch_max_appearance_cost=app_cost,
            stitch_direction_weight=direction_weight,
            stitch_max_direction_cost=direction_cost_value,
            stitch_max_jump_heads=jump,
        )
        for idx, (
            kind,
            gap,
            dist,
            ambiguity,
            speed,
            jump,
            app_weight,
            app_cost,
            direction_weight,
            direction_cost_value,
            mode,
        ) in enumerate(specs, start=1)
    ]
    return arms[:limit] if limit > 0 else arms


def ensure_video(video_path: Path, *, download_sample: bool) -> Path:
    """Ensure the input video exists, with Colab-friendly download/upload fallback."""
    if video_path.exists():
        return video_path
    if download_sample or video_path == Path("data/input_videos/sample.mp4"):
        video_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Video not found at {video_path}; downloading Pexels sample with browser headers...")
        req = urllib.request.Request(
            SAMPLE_VIDEO_URL,
            headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"},
        )
        try:
            with urllib.request.urlopen(req, timeout=90) as response, video_path.open("wb") as handle:
                shutil.copyfileobj(response, handle)
            return video_path
        except Exception as exc:
            print(f"Download failed: {exc}")

    if "google.colab" in sys.modules:
        from google.colab import files  # type: ignore

        print("Upload the input .mp4 now.")
        uploaded = files.upload()
        if uploaded:
            name = next(iter(uploaded))
            video_path.parent.mkdir(parents=True, exist_ok=True)
            Path(name).replace(video_path)
            return video_path
    raise FileNotFoundError(
        f"No video at {video_path}. In Colab, run with --download-sample or upload sample.mp4."
    )


def resolve_model(model_path: Path) -> Path:
    """Resolve a model path, with Colab upload fallback."""
    if model_path.exists() and model_path.stat().st_size > 0:
        return model_path
    candidates = [DEFAULT_RPEE_WEIGHTS, DEFAULT_CROWDHUMAN_WEIGHTS, Path("best.pt")]
    for candidate in candidates:
        if candidate.exists() and candidate.stat().st_size > 0:
            print(f"Requested model missing; using available model: {candidate}")
            return candidate
    if "google.colab" in sys.modules:
        from google.colab import files  # type: ignore

        print("Upload head detector .pt weights now.")
        uploaded = files.upload()
        if uploaded:
            name = next(iter(uploaded))
            target = Path("head_detector_best.pt")
            Path(name).replace(target)
            return target
    raise FileNotFoundError(f"Head detector weights not found: {model_path}")


def bbox_iou(a: tuple[float, float, float, float], b: tuple[float, float, float, float]) -> float:
    """Compute IoU for xyxy boxes."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def centroid(box: tuple[float, float, float, float]) -> tuple[float, float]:
    """Return box center."""
    return ((box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0)


def box_width(box: tuple[float, float, float, float]) -> float:
    """Return positive box width."""
    return max(1.0, box[2] - box[0])


def expand_box(
    box: tuple[float, float, float, float],
    factor: float,
) -> tuple[float, float, float, float]:
    """Expand a box around its center."""
    cx, cy = centroid(box)
    width, height = box[2] - box[0], box[3] - box[1]
    return (
        cx - width * factor / 2,
        cy - height * factor / 2,
        cx + width * factor / 2,
        cy + height * factor / 2,
    )


def appearance_crop_box(
    box: tuple[float, float, float, float],
    frame_shape: tuple[int, int, int],
) -> tuple[int, int, int, int]:
    """Return a head+upper-shoulder crop box clipped to the frame."""
    height, width = frame_shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    cx = (x1 + x2) / 2.0
    # Heads alone are weak appearance cues. Include a little context below the
    # head so clothing/shoulder color can disambiguate adjacent people.
    crop_w = bw * 2.2
    crop_h = bh * 3.0
    left = max(0, int(round(cx - crop_w / 2.0)))
    right = min(width, int(round(cx + crop_w / 2.0)))
    top = max(0, int(round(y1 - bh * 0.25)))
    bottom = min(height, int(round(y1 + crop_h)))
    if right <= left or bottom <= top:
        return (0, 0, 0, 0)
    return (left, top, right, bottom)


def appearance_histogram(
    frame: np.ndarray,
    box: tuple[float, float, float, float],
) -> np.ndarray | None:
    """Compute a compact HSV color histogram for head+shoulder context."""
    left, top, right, bottom = appearance_crop_box(box, frame.shape)
    if right <= left or bottom <= top:
        return None
    crop = frame[top:bottom, left:right]
    if crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [12, 8], [0, 180, 0, 256])
    hist = cv2.normalize(hist, hist).flatten().astype(np.float32)
    if not np.isfinite(hist).all() or float(hist.sum()) <= 0:
        return None
    return hist


def appearance_cost(a: np.ndarray | None, b: np.ndarray | None) -> float:
    """Return 0..1 histogram distance; 1 means unknown or very different."""
    if a is None or b is None:
        return 1.0
    similarity = float(cv2.compareHist(a.astype(np.float32), b.astype(np.float32), cv2.HISTCMP_CORREL))
    if not math.isfinite(similarity):
        return 1.0
    return max(0.0, min(1.0, (1.0 - similarity) / 2.0))


def direction_cost(a: tuple[float, float], b: tuple[float, float]) -> float:
    """Return 0..1 direction mismatch; 0 means same direction."""
    a_norm = math.hypot(a[0], a[1])
    b_norm = math.hypot(b[0], b[1])
    if a_norm < 1.0 or b_norm < 1.0:
        return 0.0
    cosine = (a[0] * b[0] + a[1] * b[1]) / max(1e-6, a_norm * b_norm)
    cosine = max(-1.0, min(1.0, cosine))
    return (1.0 - cosine) / 2.0


def nms(detections: Iterable[NormalizedDetection], iou_threshold: float) -> list[NormalizedDetection]:
    """Simple NMS over normalized detections."""
    kept: list[NormalizedDetection] = []
    for detection in sorted(detections, key=lambda item: item.confidence, reverse=True):
        if all(bbox_iou(detection.bbox, old.bbox) < iou_threshold for old in kept):
            kept.append(detection)
    return kept


class TunedHeadTracker:
    """ByteTrack wrapper with head-specific association boxes and fallback tracker."""

    def __init__(self, frame_rate: float, arm: Arm, cfg: StabilityConfig) -> None:
        self.arm = arm
        self.cfg = cfg
        self.backend = "greedy_iou"
        self.tracker = None
        if sv is not None and hasattr(sv, "ByteTrack"):
            kwargs = self._bytetrack_kwargs(frame_rate)
            try:
                self.tracker = sv.ByteTrack(**kwargs)
                self.backend = "bytetrack:" + json.dumps(kwargs, sort_keys=True)
            except TypeError:
                self.tracker = sv.ByteTrack()
                self.backend = "bytetrack:defaults"
        self._next_id = 1
        self._active: dict[int, tuple[tuple[float, float, float, float], int]] = {}

    def _bytetrack_kwargs(self, frame_rate: float) -> dict[str, object]:
        params = inspect.signature(sv.ByteTrack).parameters  # type: ignore[union-attr]
        candidates = {
            "track_activation_threshold": self.arm.activation,
            "lost_track_buffer": self.cfg.lost_track_buffer,
            "minimum_matching_threshold": self.cfg.match_thresh,
            "minimum_consecutive_frames": self.arm.consec,
            "frame_rate": max(1, int(round(frame_rate or 25))),
            # Older supervision names:
            "track_thresh": self.arm.activation,
            "track_buffer": self.cfg.lost_track_buffer,
            "match_thresh": self.cfg.match_thresh,
        }
        return {key: value for key, value in candidates.items() if key in params}

    def update(
        self,
        detections: list[NormalizedDetection],
    ) -> list[tuple[int, tuple[float, float, float, float], float]]:
        """Assign track IDs to detections."""
        if self.tracker is not None and sv is not None:
            return self._update_bytetrack(detections)
        return self._update_greedy(detections)

    def _tracking_box(
        self, detection: NormalizedDetection
    ) -> tuple[float, float, float, float]:
        return expand_box(detection.bbox, self.cfg.box_expand_factor) if self.arm.expand else detection.bbox

    def _update_bytetrack(
        self,
        detections: list[NormalizedDetection],
    ) -> list[tuple[int, tuple[float, float, float, float], float]]:
        if not detections:
            try:
                self.tracker.update_with_detections(sv.Detections.empty())  # type: ignore[union-attr]
            except Exception:
                pass
            return []

        track_boxes = np.array([self._tracking_box(detection) for detection in detections], dtype=float)
        confidence = np.array([detection.confidence for detection in detections], dtype=float)
        class_id = np.zeros(len(detections), dtype=int)
        original = np.array([detection.bbox for detection in detections], dtype=float)
        data = {
            "ox1": original[:, 0],
            "oy1": original[:, 1],
            "ox2": original[:, 2],
            "oy2": original[:, 3],
        }
        sv_detections = sv.Detections(  # type: ignore[union-attr]
            xyxy=track_boxes,
            confidence=confidence,
            class_id=class_id,
            data=data,
        )
        tracked = self.tracker.update_with_detections(sv_detections)
        if tracked.tracker_id is None:
            return []

        results: list[tuple[int, tuple[float, float, float, float], float]] = []
        for idx in range(len(tracked)):
            original_box = (
                float(tracked.data["ox1"][idx]),
                float(tracked.data["oy1"][idx]),
                float(tracked.data["ox2"][idx]),
                float(tracked.data["oy2"][idx]),
            )
            conf = float(tracked.confidence[idx]) if tracked.confidence is not None else 0.0
            results.append((int(tracked.tracker_id[idx]), original_box, conf))
        return results

    def _update_greedy(
        self,
        detections: list[NormalizedDetection],
    ) -> list[tuple[int, tuple[float, float, float, float], float]]:
        for track_id in list(self._active):
            box, lost = self._active[track_id]
            self._active[track_id] = (box, lost + 1)

        used: set[int] = set()
        results: list[tuple[int, tuple[float, float, float, float], float]] = []
        for detection in sorted(detections, key=lambda item: item.confidence, reverse=True):
            track_box = self._tracking_box(detection)
            best_id: int | None = None
            best_iou = 0.0
            for track_id, (box, lost) in self._active.items():
                if track_id in used or lost > self.cfg.lost_track_buffer:
                    continue
                score = bbox_iou(track_box, box)
                if score > best_iou:
                    best_id, best_iou = track_id, score
            if best_id is None or best_iou < max(0.10, 1.0 - self.cfg.match_thresh):
                best_id = self._next_id
                self._next_id += 1
            used.add(best_id)
            self._active[best_id] = (track_box, 0)
            results.append((best_id, detection.bbox, detection.confidence))

        for track_id in list(self._active):
            if self._active[track_id][1] > self.cfg.lost_track_buffer:
                del self._active[track_id]
        return results


def stitch_tracks(
    per_frame: list[list[tuple[int, tuple[float, float, float, float], float]]],
    *,
    gap_frames: int,
    dist_heads: float,
    mode: str = "spatial",
    ambiguity_ratio: float = 0.80,
    max_speed_heads: float = 0.45,
    appearance_by_track: dict[int, dict[str, np.ndarray | None]] | None = None,
    appearance_weight: float = 0.0,
    max_appearance_cost: float = 1.0,
    direction_weight: float = 0.0,
    max_direction_cost: float = 1.0,
    max_jump_heads: float = 0.0,
) -> dict[int, int]:
    """Merge short-gap track fragments by spatial continuity."""
    info: dict[int, dict[str, object]] = {}
    for frame_index, detections in enumerate(per_frame):
        for track_id, box, _conf in detections:
            if track_id not in info:
                info[track_id] = {
                    "first": frame_index,
                    "last": frame_index,
                    "first_c": centroid(box),
                    "last_c": centroid(box),
                    "centers": [],
                    "frames": set(),
                    "widths": [box_width(box)],
                }
            row = info[track_id]
            row["last"] = frame_index
            row["last_c"] = centroid(box)
            row["centers"].append((frame_index, centroid(box)))  # type: ignore[union-attr]
            row["frames"].add(frame_index)  # type: ignore[union-attr]
            row["widths"].append(box_width(box))  # type: ignore[union-attr]

    parent = {track_id: track_id for track_id in info}
    component_frames = {track_id: set(row["frames"]) for track_id, row in info.items()}

    def find(track_id: int) -> int:
        while parent[track_id] != track_id:
            parent[track_id] = parent[parent[track_id]]
            track_id = parent[track_id]
        return track_id

    def union(earlier: int, later: int) -> None:
        root_a, root_b = find(earlier), find(later)
        if root_a != root_b:
            if component_frames[root_a] & component_frames[root_b]:
                return
            parent[root_b] = root_a
            component_frames[root_a].update(component_frames[root_b])

    def velocity(track_id: int, *, tail: bool) -> tuple[float, float]:
        centers = info[track_id]["centers"]  # type: ignore[assignment]
        if len(centers) < 2:
            return (0.0, 0.0)
        segment = centers[-6:] if tail else centers[:6]
        if len(segment) < 2:
            return (0.0, 0.0)
        f0, c0 = segment[0]
        f1, c1 = segment[-1]
        dt = max(1, int(f1) - int(f0))
        return ((c1[0] - c0[0]) / dt, (c1[1] - c0[1]) / dt)

    births = sorted(info, key=lambda track_id: int(info[track_id]["first"]))
    for born_id in births:
        born = info[born_id]
        born_frame = int(born["first"])
        born_center = born["first_c"]  # type: ignore[assignment]
        born_width = float(np.median(born["widths"]))  # type: ignore[arg-type]
        candidates: list[tuple[float, int]] = []
        for old_id, old in info.items():
            if old_id == born_id:
                continue
            gap = born_frame - int(old["last"])
            if gap <= 0 or gap > gap_frames:
                continue
            old_center = old["last_c"]  # type: ignore[assignment]
            raw_distance = math.hypot(born_center[0] - old_center[0], born_center[1] - old_center[1])
            max_head_width = max(born_width, float(np.median(old["widths"])))  # type: ignore[arg-type]
            raw_jump_heads = raw_distance / max(1.0, max_head_width)
            if max_jump_heads > 0.0 and raw_jump_heads > max_jump_heads:
                continue
            distance = raw_distance
            direction_mismatch = 0.0
            if mode in {"velocity", "observation"}:
                vx, vy = velocity(old_id, tail=True)
                predicted = (old_center[0] + vx * gap, old_center[1] + vy * gap)
                predicted_distance = math.hypot(
                    born_center[0] - predicted[0],
                    born_center[1] - predicted[1],
                )
                distance = min(distance, predicted_distance)
                if mode == "observation":
                    bvx, bvy = velocity(born_id, tail=False)
                    backward = (born_center[0] - bvx * gap, born_center[1] - bvy * gap)
                    backward_distance = math.hypot(
                        old_center[0] - backward[0],
                        old_center[1] - backward[1],
                    )
                    if math.hypot(vx, vy) >= 1.0 and math.hypot(bvx, bvy) >= 1.0:
                        distance = min(distance, max(predicted_distance, backward_distance))
                    direction_mismatch = direction_cost((vx, vy), (bvx, bvy))
                    if direction_mismatch > max_direction_cost:
                        continue
            threshold = dist_heads * max_head_width
            implied_speed_heads = distance / max(1, gap) / max(1.0, born_width)
            if distance <= threshold and implied_speed_heads <= max_speed_heads:
                spatial_cost = distance / max(1.0, threshold)
                app_cost = 0.0
                if appearance_weight > 0.0:
                    old_hist = (appearance_by_track or {}).get(old_id, {}).get("last")
                    born_hist = (appearance_by_track or {}).get(born_id, {}).get("first")
                    app_cost = appearance_cost(old_hist, born_hist)
                    if app_cost > max_appearance_cost:
                        continue
                motion_weight = max(0.0, 1.0 - appearance_weight - direction_weight)
                total_cost = (
                    motion_weight * spatial_cost
                    + appearance_weight * app_cost
                    + direction_weight * direction_mismatch
                )
                candidates.append((total_cost, old_id))
        if not candidates:
            continue
        candidates.sort(key=lambda item: item[0])
        best_score, best_id = candidates[0]
        if len(candidates) > 1:
            second_score = candidates[1][0]
            # If the best candidate is not clearly better, do not stitch. Dense
            # crowds often have several plausible nearby heads; a skipped stitch
            # is safer than assigning one ID to two different people.
            if second_score > 0 and best_score / second_score > ambiguity_ratio:
                continue
        if best_id is not None:
            union(best_id, born_id)
    return {track_id: find(track_id) for track_id in info}


def video_meta(path: Path) -> tuple[float, int, int, int]:
    """Return fps, width, height, frame count."""
    capture = cv2.VideoCapture(str(path))
    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()
    return fps, width, height, frames


def open_writer(path: Path, fps: float, size: tuple[int, int]) -> cv2.VideoWriter:
    """Open a Colab-safe video writer."""
    path.parent.mkdir(parents=True, exist_ok=True)
    for codec in ("mp4v", "MJPG", "XVID"):
        writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), fps, size)
        if writer.isOpened():
            return writer
    raise RuntimeError(f"Cannot open VideoWriter for {path}")


def color_for(track_id: int) -> tuple[int, int, int]:
    """Deterministic BGR color by track ID."""
    raw = (int(track_id) * 2654435761) % (256**3)
    return (raw % 256, (raw // 256) % 256, (raw // 65536) % 256)


def model_label_for(path: Path, override: str | None) -> str:
    """Build a compact model label for output paths."""
    if override:
        return override
    if path == DEFAULT_RPEE_WEIGHTS:
        return "rpee_head_s"
    if path == DEFAULT_CROWDHUMAN_WEIGHTS:
        return "crowdhuman_head_s"
    return path.stem.replace(".", "_")


def run_arm(
    arm: Arm,
    detector: Detector,
    video_path: Path,
    out_dir: Path,
    cfg: StabilityConfig,
    *,
    max_frames: int,
    draw_annotated: bool,
) -> dict[str, object]:
    """Run one arm and return summary metrics."""
    fps, width, height, total_frames = video_meta(video_path)
    if max_frames > 0:
        total_frames = min(total_frames, max_frames)
    detector.confidence = arm.conf
    tracker = TunedHeadTracker(fps, arm, cfg)
    stitch_gap_frames = arm.stitch_gap_frames or cfg.stitch_gap_frames
    stitch_dist_heads = arm.stitch_dist_heads or cfg.stitch_dist_heads
    stitch_mode = arm.stitch_mode or cfg.stitch_mode
    stitch_ambiguity_ratio = arm.stitch_ambiguity_ratio or cfg.stitch_ambiguity_ratio
    stitch_max_speed_heads = arm.stitch_max_speed_heads or cfg.stitch_max_speed_heads
    stitch_appearance_weight = (
        arm.stitch_appearance_weight
        if arm.stitch_appearance_weight is not None
        else cfg.stitch_appearance_weight
    )
    stitch_max_appearance_cost = (
        arm.stitch_max_appearance_cost
        if arm.stitch_max_appearance_cost is not None
        else cfg.stitch_max_appearance_cost
    )
    stitch_direction_weight = (
        arm.stitch_direction_weight
        if arm.stitch_direction_weight is not None
        else cfg.stitch_direction_weight
    )
    stitch_max_direction_cost = (
        arm.stitch_max_direction_cost
        if arm.stitch_max_direction_cost is not None
        else cfg.stitch_max_direction_cost
    )
    stitch_max_jump_heads = (
        arm.stitch_max_jump_heads if arm.stitch_max_jump_heads is not None else cfg.stitch_max_jump_heads
    )

    per_frame: list[list[tuple[int, tuple[float, float, float, float], float]]] = []
    appearance_by_track: dict[int, dict[str, np.ndarray | None]] = {}
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    with tqdm(total=total_frames if total_frames > 0 else None, desc=f"{arm.name} track") as progress:
        while True:
            if max_frames > 0 and frame_index >= max_frames:
                break
            ok, frame = capture.read()
            if not ok:
                break
            raw = detector.detect(frame, frame_index=frame_index, timestamp=frame_index / fps).detections
            raw = nms(raw, cfg.head_nms_iou)
            tracked = tracker.update(raw)
            for track_id, box, _conf in tracked:
                hist = appearance_histogram(frame, box) if stitch_appearance_weight > 0.0 else None
                if track_id not in appearance_by_track:
                    appearance_by_track[track_id] = {"first": hist, "last": hist}
                elif hist is not None:
                    appearance_by_track[track_id]["last"] = hist
            per_frame.append(tracked)
            frame_index += 1
            progress.update(1)
    capture.release()

    remap = (
        stitch_tracks(
            per_frame,
            gap_frames=stitch_gap_frames,
            dist_heads=stitch_dist_heads,
            mode=stitch_mode,
            ambiguity_ratio=stitch_ambiguity_ratio,
            max_speed_heads=stitch_max_speed_heads,
            appearance_by_track=appearance_by_track,
            appearance_weight=stitch_appearance_weight,
            max_appearance_cost=stitch_max_appearance_cost,
            direction_weight=stitch_direction_weight,
            max_direction_cost=stitch_max_direction_cost,
            max_jump_heads=stitch_max_jump_heads,
        )
        if arm.stitch
        else {}
    )

    def root_id(track_id: int) -> int:
        return remap.get(track_id, track_id)

    stats: dict[int, dict[str, object]] = defaultdict(lambda: {"frames": set(), "boxes": []})
    for frame_number, detections in enumerate(per_frame):
        for track_id, box, _conf in detections:
            rid = root_id(track_id)
            stats[rid]["frames"].add(frame_number)  # type: ignore[union-attr]
            stats[rid]["boxes"].append((frame_number, box))  # type: ignore[union-attr]

    visible = {rid: len(row["frames"]) for rid, row in stats.items()}
    confirmed = {rid for rid, count in visible.items() if count >= cfg.min_confirmed_age}
    first = {rid: min(row["frames"]) for rid, row in stats.items() if row["frames"]}  # type: ignore[arg-type]
    last = {rid: max(row["frames"]) for rid, row in stats.items() if row["frames"]}  # type: ignore[arg-type]

    per_frame_counts = []
    duplicate_id_frames = 0
    duplicate_id_instances = 0
    max_same_id_instances = 1
    for detections in per_frame:
        frame_ids = [root_id(track_id) for track_id, _box, _conf in detections if root_id(track_id) in confirmed]
        unique_frame_ids = set(frame_ids)
        if len(frame_ids) != len(unique_frame_ids):
            duplicate_id_frames += 1
            duplicate_id_instances += len(frame_ids) - len(unique_frame_ids)
            for frame_id in unique_frame_ids:
                max_same_id_instances = max(max_same_id_instances, frame_ids.count(frame_id))
        per_frame_counts.append(len(unique_frame_ids))

    raw_unique = len({track_id for detections in per_frame for track_id, _box, _conf in detections})
    unique_poststitch = len(stats)
    confirmed_unique = len(confirmed)
    peak_concurrent = max(per_frame_counts) if per_frame_counts else 0
    median_concurrent = float(np.median(per_frame_counts)) if per_frame_counts else 0.0
    short_lived = sum(1 for count in visible.values() if count < cfg.min_confirmed_age)
    median_lifespan = float(np.median([visible[rid] for rid in confirmed])) if confirmed else 0.0
    inflation = confirmed_unique / peak_concurrent if peak_concurrent else float("nan")

    first_center = {rid: centroid(stats[rid]["boxes"][0][1]) for rid in confirmed}  # type: ignore[index]
    last_center = {rid: centroid(stats[rid]["boxes"][-1][1]) for rid in confirmed}  # type: ignore[index]
    switch_events = 0
    for rid in confirmed:
        for other in confirmed:
            if other == rid:
                continue
            gap = first[rid] - last[other]
            if 0 < gap <= cfg.switch_lookback:
                distance = math.hypot(
                    first_center[rid][0] - last_center[other][0],
                    first_center[rid][1] - last_center[other][1],
                )
                first_box = stats[rid]["boxes"][0][1]  # type: ignore[index]
                if distance <= stitch_dist_heads * box_width(first_box):
                    switch_events += 1
                    break

    jump_events = 0
    max_gap_jump_heads = 0.0
    for rid in confirmed:
        observations = sorted(stats[rid]["boxes"], key=lambda item: item[0])  # type: ignore[arg-type]
        for (prev_frame, prev_box), (next_frame, next_box) in zip(observations, observations[1:]):
            gap = int(next_frame) - int(prev_frame)
            if gap <= 1:
                continue
            distance = math.hypot(
                centroid(next_box)[0] - centroid(prev_box)[0],
                centroid(next_box)[1] - centroid(prev_box)[1],
            )
            distance_heads = distance / max(1.0, box_width(prev_box), box_width(next_box))
            max_gap_jump_heads = max(max_gap_jump_heads, distance_heads)
            if gap <= stitch_gap_frames and distance_heads > stitch_dist_heads:
                jump_events += 1

    gt_errors = []
    for frame_id, gt_count in MANUAL_GT.items():
        if frame_id < len(per_frame_counts):
            gt_errors.append(abs(per_frame_counts[frame_id] - gt_count))
    gt_mae = float(np.mean(gt_errors)) if gt_errors else float("nan")

    per_frame_csv = out_dir / f"{arm.name}_per_frame.csv"
    pd.DataFrame(
        {
            "frame": list(range(len(per_frame_counts))),
            "current_visible_confirmed": per_frame_counts,
        }
    ).to_csv(per_frame_csv, index=False)

    lifespans_csv = out_dir / f"{arm.name}_lifespans.csv"
    pd.DataFrame(
        [
            {
                "track_id": rid,
                "visible_frames": visible[rid],
                "first_frame": first.get(rid),
                "last_frame": last.get(rid),
                "confirmed": rid in confirmed,
            }
            for rid in sorted(stats)
        ]
    ).to_csv(lifespans_csv, index=False)

    annotated_video = out_dir / f"{arm.name}_annotated.mp4"
    if draw_annotated:
        capture = cv2.VideoCapture(str(video_path))
        writer = open_writer(annotated_video, fps, (width, height))
        frame_index = 0
        with tqdm(total=len(per_frame), desc=f"{arm.name} draw") as progress:
            while frame_index < len(per_frame):
                ok, frame = capture.read()
                if not ok:
                    break
                current_visible = 0
                for track_id, box, _conf in per_frame[frame_index]:
                    rid = root_id(track_id)
                    if rid not in confirmed:
                        continue
                    current_visible += 1
                    x1, y1, x2, y2 = (int(value) for value in box)
                    color = color_for(rid)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(
                        frame,
                        str(rid),
                        (x1, max(10, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        color,
                        1,
                        cv2.LINE_AA,
                    )
                labels = [
                    arm.name,
                    f"visible(confirmed): {current_visible}",
                    f"unique(confirmed): {confirmed_unique}",
                    f"raw IDs: {raw_unique}",
                ]
                for idx, label in enumerate(labels):
                    y = 24 + idx * 24
                    cv2.putText(frame, label, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 4, cv2.LINE_AA)
                    cv2.putText(frame, label, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
                writer.write(frame)
                frame_index += 1
                progress.update(1)
        capture.release()
        writer.release()
    annotated_video_text = str(annotated_video) if draw_annotated else ""

    return {
        "arm": arm.name,
        "conf": arm.conf,
        "backend": tracker.backend,
        "stitch_gap_frames": stitch_gap_frames if arm.stitch else 0,
        "stitch_dist_heads": stitch_dist_heads if arm.stitch else 0,
        "stitch_mode": stitch_mode if arm.stitch else "none",
        "stitch_ambiguity_ratio": stitch_ambiguity_ratio if arm.stitch else 0,
        "stitch_max_speed_heads": stitch_max_speed_heads if arm.stitch else 0,
        "stitch_appearance_weight": stitch_appearance_weight if arm.stitch else 0,
        "stitch_max_appearance_cost": stitch_max_appearance_cost if arm.stitch else 0,
        "stitch_direction_weight": stitch_direction_weight if arm.stitch else 0,
        "stitch_max_direction_cost": stitch_max_direction_cost if arm.stitch else 0,
        "stitch_max_jump_heads": stitch_max_jump_heads if arm.stitch else 0,
        "peak_concurrent_confirmed": peak_concurrent,
        "median_concurrent_confirmed": round(median_concurrent, 2),
        "raw_unique_ids_prestitch": raw_unique,
        "unique_ids_poststitch": unique_poststitch,
        "confirmed_unique": confirmed_unique,
        "inflation_factor": round(inflation, 2) if inflation == inflation else None,
        "duplicate_id_frames": duplicate_id_frames,
        "duplicate_id_instances": duplicate_id_instances,
        "max_same_id_instances": max_same_id_instances,
        "residual_switch_events": switch_events,
        "gap_jump_events": jump_events,
        "max_gap_jump_heads": round(max_gap_jump_heads, 2),
        "short_lived_tracks": short_lived,
        "median_confirmed_lifespan": round(median_lifespan, 2),
        "gt_mae": round(gt_mae, 2) if gt_mae == gt_mae else None,
        "per_frame_csv": str(per_frame_csv),
        "lifespans_csv": str(lifespans_csv),
        "annotated_video": annotated_video_text,
    }


def make_side_by_side(summaries: list[dict[str, object]], out_path: Path, target_height: int = 480) -> None:
    """Render annotated videos side-by-side for visual review."""
    caps = [cv2.VideoCapture(str(summary["annotated_video"])) for summary in summaries]
    fps = caps[0].get(cv2.CAP_PROP_FPS) or 25.0
    sizes = []
    for cap in caps:
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        sizes.append((max(1, int(width * target_height / max(1, height))), target_height))
    writer = open_writer(out_path, fps, (sum(width for width, _height in sizes), target_height))
    while True:
        frames = []
        for cap, (width, height) in zip(caps, sizes):
            ok, frame = cap.read()
            if not ok:
                frames = None
                break
            frames.append(cv2.resize(frame, (width, height)))
        if frames is None:
            break
        writer.write(cv2.hconcat(frames))
    for cap in caps:
        cap.release()
    writer.release()


def ranked_metrics(table: pd.DataFrame) -> pd.DataFrame:
    """Rank arms by safety gates first, then by unique-ID compression."""
    ranked = table.copy()
    duplicate_penalty = (
        ranked["duplicate_id_frames"] * 500
        + ranked["duplicate_id_instances"] * 1000
        + (ranked["max_same_id_instances"] - 1).clip(lower=0) * 1000
    )
    stability_penalty = (
        ranked["gap_jump_events"] * 3
        + ranked["max_gap_jump_heads"] * 2
        + ranked["residual_switch_events"] * 4
    )
    ranked["passes_hard_gates"] = (
        (ranked["duplicate_id_frames"] == 0)
        & (ranked["duplicate_id_instances"] == 0)
        & (ranked["max_same_id_instances"] == 1)
    )
    ranked["selection_score"] = (
        ranked["confirmed_unique"]
        + duplicate_penalty
        + stability_penalty
        + ranked["gt_mae"].fillna(0)
    ).round(2)
    return ranked.sort_values(
        ["passes_hard_gates", "selection_score", "confirmed_unique"],
        ascending=[False, True, True],
    )


def main() -> int:
    """Run selected arms and save outputs."""
    args = parse_args()
    video_path = ensure_video(args.video, download_sample=args.download_sample)
    model_path = resolve_model(args.model)
    label = model_label_for(model_path, args.model_label)
    cfg = StabilityConfig(imgsz=args.imgsz, stitch_mode=args.stitch_mode)
    generated_arms: list[Arm] = []
    for profile in args.sweep_profiles:
        generated_arms.extend(sweep_arms(profile, limit=args.sweep_limit))

    available = {arm.name: arm for arm in (*DEFAULT_ARMS, *generated_arms)}
    requested_names = args.arms or ["baseline", "fix_0p16", "fix_0p30"]
    unknown_names = [name for name in requested_names if name not in available]
    if unknown_names:
        print(f"Unknown arm name(s): {unknown_names}")
        print("Available arms:")
        for name in sorted(available):
            print(f"  {name}")
        return 2

    arms = [available[name] for name in requested_names]
    requested_sweep_arm = any(name.startswith("sweep_") for name in requested_names)
    if args.sweep_profiles and not requested_sweep_arm:
        requested_set = set(requested_names)
        arms.extend(arm for arm in generated_arms if arm.name not in requested_set)

    out_dir = args.output_root / video_path.stem / label
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Video: {video_path}")
    print(f"Model: {model_path}")
    print(f"Output: {out_dir}")
    print(f"Arms: {[arm.name for arm in arms]}")

    detector = Detector(
        weights_path=str(model_path),
        device=args.device,
        confidence=0.16,
        iou=cfg.detector_iou,
        imgsz=cfg.imgsz,
        max_det=cfg.max_det,
        half=args.half,
        use_fine_tuned_if_available=False,
        detector_mode="head",
    )

    summaries = [
        run_arm(
            arm,
            detector,
            video_path,
            out_dir,
            cfg,
            max_frames=args.max_frames,
            draw_annotated=args.draw_annotated,
        )
        for arm in arms
    ]
    table = pd.DataFrame(summaries)
    metrics_path = out_dir / "comparison_metrics.csv"
    table.to_csv(metrics_path, index=False)
    ranked = ranked_metrics(table)
    ranked_path = out_dir / "ranked_metrics.csv"
    ranked.to_csv(ranked_path, index=False)
    print("\n=== comparison_metrics ===")
    print(table[
        [
            "arm",
            "stitch_gap_frames",
            "stitch_dist_heads",
            "stitch_mode",
            "stitch_ambiguity_ratio",
            "stitch_max_speed_heads",
            "stitch_appearance_weight",
            "stitch_max_appearance_cost",
            "stitch_direction_weight",
            "stitch_max_direction_cost",
            "stitch_max_jump_heads",
            "peak_concurrent_confirmed",
            "median_concurrent_confirmed",
            "raw_unique_ids_prestitch",
            "unique_ids_poststitch",
            "confirmed_unique",
            "inflation_factor",
            "duplicate_id_frames",
            "duplicate_id_instances",
            "max_same_id_instances",
            "residual_switch_events",
            "gap_jump_events",
            "max_gap_jump_heads",
            "gt_mae",
        ]
    ].to_string(index=False))
    print(f"\nSaved: {metrics_path}")
    print(f"Ranked: {ranked_path}")

    print("\n=== top_ranked_candidates ===")
    print(
        ranked[
            [
                "arm",
                "passes_hard_gates",
                "selection_score",
                "confirmed_unique",
                "residual_switch_events",
                "gap_jump_events",
                "max_gap_jump_heads",
                "median_concurrent_confirmed",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

    if args.draw_annotated and args.make_side_by_side and len(summaries) > 1:
        side_path = out_dir / "side_by_side.mp4"
        make_side_by_side(summaries, side_path)
        print(f"Side-by-side: {side_path}")
    elif args.make_side_by_side and not args.draw_annotated:
        print("Side-by-side skipped because --no-draw-annotated was set.")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Pin versions; can be relaxed to ultralytics>=8.4 supervision>=0.28 if pip complains
%pip install -q "ultralytics==8.4.58" "supervision==0.28.0" "tqdm>=4.66"
import ultralytics, supervision
print("ultralytics", ultralytics.__version__, "| supervision", supervision.__version__)


## Step 2 — get the model + sample video (from the v7-handoff release)


In [ ]:
import os, urllib.request, pathlib

BASE = "https://github.com/Sriniketh24/Crowd-analysis/releases/download/v7-handoff"
assets = {
    "models/fine_tuned/head_detector_s/weights/best.pt": f"{BASE}/head_detector_s_best.pt",
    "data/input_videos/sample.mp4": f"{BASE}/sample.mp4",
}
for dest_rel, url in assets.items():
    dest = pathlib.Path(dest_rel)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  already present: {dest_rel} ({dest.stat().st_size / 1e6:.1f} MB)")
        continue
    print(f"  downloading {dest_rel} …")
    urllib.request.urlretrieve(url, dest)
    print(f"  → {dest.stat().st_size / 1e6:.1f} MB")


## Step 3 — run the pipeline on the GPU


In [ ]:
# To run only the winner: --arms fix_0p16_wide_guard_app
!python scripts/run_head_id_stability.py --video data/input_videos/sample.mp4 --model models/fine_tuned/head_detector_s/weights/best.pt --arms baseline fix_0p16_wide_guard_app --device cuda --half --imgsz 1536 --output-root data/outputs/handoff_run


In [ ]:
import pandas as pd
m = pd.read_csv("data/outputs/handoff_run/sample/rpee_head_s/comparison_metrics.csv")
cols = ["arm", "confirmed_unique", "inflation_factor", "peak_concurrent_confirmed",
        "residual_switch_events", "duplicate_id_frames", "gt_mae"]
m[cols]


In [ ]:
from google.colab import files
OUT = "data/outputs/handoff_run/sample/rpee_head_s"
v7 = f"{OUT}/fix_0p16_wide_guard_app_annotated.mp4"
print("v7 annotated video:", v7)
files.download(v7)
# also: files.download(f"{OUT}/side_by_side.mp4")  for the comparison


## Optional: persist outputs to Google Drive

Mount Google Drive and copy the run outputs so they survive when the runtime recycles.


In [ ]:
from google.drive import drive  # type: ignore
drive.mount('/content/drive')
import shutil, pathlib
src = pathlib.Path('data/outputs/handoff_run')
dst = pathlib.Path('/content/drive/MyDrive/crowd-analysis-outputs/handoff_run')
shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
print('Saved to:', dst)


## Notes & troubleshooting

- **All arms** are defined in the embedded `scripts/run_head_id_stability.py` under `DEFAULT_ARMS`.
  The v7 winner is `fix_0p16_wide_guard_app`.
- **Your own video**: change `--video` to any `.mp4` path; note that `gt_mae`/`MANUAL_GT`
  is calibrated to `sample.mp4` only — ignore it for other videos.
- **Speed tips**: use `--imgsz 1280` or add `--max-frames 200` for a quick preview.
- **Pip pin fallback**: if pip refuses the exact pins, relax to
  `ultralytics>=8.4 supervision>=0.28`.
- This notebook is generated — to update it, edit the pipeline in the repo and re-run
  `python scripts/build_handoff_notebook.py` (commit `c992dc0`).
